# PlasticWise AI - 00_setup_and_download
**Week 1:** GPU check, Drive folders, Kaggle dataset download, asal image counts.

Cells ko upar se neeche **ek ek karke** run karo. Pehle: Runtime > Change runtime type > **T4 GPU**.

## Cell 1: GPU check aur libraries

In [ ]:
import tensorflow as tf
import numpy as np, pandas as pd
import matplotlib, seaborn, sklearn, PIL

print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPU:", gpus if gpus else "NOT FOUND -> Runtime > Change runtime type > T4 GPU")

## Cell 2: Google Drive mount aur project folders

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/PlasticWise'
for sub in ['data', 'results', 'models', 'app']:
    os.makedirs(f'{BASE}/{sub}', exist_ok=True)
print("Folders ready:", os.listdir(BASE))

## Cell 3: Kaggle dataset download
Public dataset hai, aam taur par login ke baghair download ho jata hai. Agar login maange to Cell 3b chalao.

In [ ]:
import kagglehub
path = kagglehub.dataset_download("sumn2u/garbage-classification-v2")
print("Downloaded to:", path)

### Cell 3b (sirf agar login error aaye)
Kaggle > Settings > API > *Create New Token* se username/key lo.

In [ ]:
# import kagglehub
# kagglehub.login()   # username aur key yahan paste karo
# path = kagglehub.dataset_download("sumn2u/garbage-classification-v2")
# print(path)

## Cell 4: Class folders dhoondo aur folder structure dekho

In [ ]:
from pathlib import Path

IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def find_data_dir(root):
    """Wo folder dhoondo jiske andar sab se zyada class sub-folders hon (jinme images hon)."""
    best, best_n = None, 0
    for d in [Path(root)] + [p for p in Path(root).rglob('*') if p.is_dir()]:
        classes = [s for s in d.iterdir() if s.is_dir() and any(f.suffix.lower() in IMG_EXT for f in s.iterdir() if f.is_file())]
        if len(classes) > best_n:
            best, best_n = d, len(classes)
    return best

DATA_DIR = find_data_dir(path)
print("DATA_DIR =", DATA_DIR)
print("Classes  =", sorted(p.name for p in DATA_DIR.iterdir() if p.is_dir()))

## Cell 5: Har class ki asal image count
**Ye numbers report aur assignment mein likhne hain** (proposal wala 19,762 copy mat karo).

In [ ]:
rows = []
for cls in sorted(p for p in DATA_DIR.iterdir() if p.is_dir()):
    n = sum(1 for f in cls.rglob('*') if f.is_file() and f.suffix.lower() in IMG_EXT)
    rows.append({'class': cls.name, 'images': n})

counts = pd.DataFrame(rows)
counts['percent'] = (counts['images'] / counts['images'].sum() * 100).round(2)
print(counts.to_string(index=False))
print("\nTOTAL images :", counts['images'].sum())
print("TOTAL classes:", len(counts))

counts.to_csv(f'{BASE}/results/class_counts_raw.csv', index=False)
print("Saved -> results/class_counts_raw.csv")

## Cell 6: Aik sample image check

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import random

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, cls in zip(axes.ravel(), sorted(p for p in DATA_DIR.iterdir() if p.is_dir())):
    imgs = [f for f in cls.rglob('*') if f.suffix.lower() in IMG_EXT]
    im = Image.open(random.choice(imgs)).convert('RGB')
    ax.imshow(im); ax.set_title(f"{cls.name} {im.size}"); ax.axis('off')
plt.tight_layout(); plt.show()

## Cell 7: DATA_DIR yaad rakho
Colab session band hone par dataset local disk se ghayab ho jata hai. Agle notebooks ke shuru mein **Cell 3 aur 4 dobara chalao** (download jaldi ho jata hai).

### Week 1 checklist
- [ ] Har class ki asal count upar table mein dekh li
- [ ] `class_counts_raw.csv` Drive mein save hui
- [ ] Notebook GitHub repo mein `notebooks/` folder mein commit karo

In [ ]:
with open(f'{BASE}/data/DATA_DIR.txt', 'w') as f:
    f.write(str(DATA_DIR))
print("Saved DATA_DIR note.")